In [ ]:
from pathlib import Path
import pandas as pd

import teehr
from teehr import RemoteReadWriteEvaluation

from setup_utils import create_minio_spark_session

### Create spark and init eval

In [ ]:
spark = create_minio_spark_session()

In [ ]:
ev = RemoteReadWriteEvaluation(spark=spark, enable_spark_proxy=True)

### Load JTS to the warehouse

In [ ]:
# fix timestamp issue
def make_parquet_spark_compatible(path: Path, output_path: Path | None = None) -> Path:
    """Rewrite parquet files with nanosecond timestamps into a Spark-compatible format."""
    import pyarrow as pa
    import pyarrow.parquet as pq
    import shutil
    
    table = pq.read_table(path)

    if output_path is None:
        output_path = path.with_suffix('.spark.parquet')

    if output_path.exists():
        if output_path.is_dir():
            shutil.rmtree(output_path)
        else:
            output_path.unlink()

    converted_fields = []
    for field in table.schema:
        if isinstance(field.type, pa.TimestampType) and field.type.unit == 'ns':
            converted_fields.append(pa.field(field.name, pa.timestamp('us'), nullable=field.nullable))
        else:
            converted_fields.append(field)

    if converted_fields != list(table.schema):
        table = table.cast(pa.schema(converted_fields))

    pq.write_table(table, output_path)
    return output_path

inputs_dir = Path(Path.cwd(), 'FIRO_data')

# load joined_timeseries from parquet to spark dataframe
joined_timeseries_path = Path(inputs_dir, 'joined_timeseries.parquet')

output_path = Path(inputs_dir, 'joined_timeseries_fixed.parquet')
compatible_path = make_parquet_spark_compatible(
    path=joined_timeseries_path,
    output_path=output_path
)
sdf = spark.read.parquet(str(compatible_path))

In [ ]:
ev._write.to_warehouse(
    source_data=sdf,
    table_name='joined_timeseries',
    write_mode="create_or_replace"
)

### Kill spark

In [ ]:
spark.stop()